# N-Queens: Four Algorithms Benchmark

This notebook implements and benchmarks four algorithms for the N-Queens problem:

1. **Exhaustive DFS / Backtracking** with column + diagonal sets and timeout
2. **Greedy Hill Climbing** using Min-Conflicts with random restarts
3. **Simulated Annealing** with incremental conflict delta updates
4. **Genetic Algorithm** with permutation representation, ordered crossover, swap mutation, elitism, and tournament selection

All conflict calculations are O(N) using column and diagonal count arrays.
Board representation: `board[row] = column`.
Random seed is fixed for reproducibility.


In [ ]:
import gc
import math
import random
import sys
import time
import tracemalloc

import matplotlib.pyplot as plt
import pandas as pd

SEED = 42
N_VALUES = [10, 30, 50, 100, 200, 500]
# Per-(algorithm, N) safety cap — generous, never trips on the optimised
# algorithms below. With four algorithms and six N values, a 240 s ceiling
# bounds the whole sweep at 24·240 s = ~96 min in the absolute worst case,
# while empirically the run completes in a couple of minutes.
TIMEOUT = 240.0
WARM_START_N = 100  # SA uses min-conflicts warm start when N >= this

random.seed(SEED)
print(f"Seed={SEED}, N values={N_VALUES}, safety_cap={TIMEOUT}s, "
      f"warm_start_N={WARM_START_N}")


## Shared bookkeeping: O(N) conflict counts

We maintain three counters: `col[c]`, `d1[r+c]` for the "/" diagonals, and `d2[r-c+N-1]`
for the "\\" diagonals. Total conflicts is `sum_k C(k, 2)` over each counter. Updates are
O(1) per queen move using the identity `C(k+1,2) - C(k,2) = k` and `C(k-1,2) - C(k,2) = -(k-1)`.


In [ ]:
def init_counts(board, N):
    """Build column and diagonal count arrays in O(N)."""
    col = [0] * N
    d1 = [0] * (2 * N)
    d2 = [0] * (2 * N)
    for r in range(N):
        c = board[r]
        col[c] += 1
        d1[r + c] += 1
        d2[r - c + N - 1] += 1
    return col, d1, d2


def total_conflicts(col, d1, d2):
    s = 0
    for k in col:
        if k > 1:
            s += k * (k - 1) // 2
    for k in d1:
        if k > 1:
            s += k * (k - 1) // 2
    for k in d2:
        if k > 1:
            s += k * (k - 1) // 2
    return s


def total_conflicts_perm(d1, d2):
    """Permutation rep: only diagonal conflicts."""
    s = 0
    for k in d1:
        if k > 1:
            s += k * (k - 1) // 2
    for k in d2:
        if k > 1:
            s += k * (k - 1) // 2
    return s


def apply_swap(board, i, j, d1, d2, N):
    """Swap board[i] and board[j], update d1/d2 in place, return conflict delta."""
    ci, cj = board[i], board[j]
    delta = 0
    # Remove (i, ci)
    delta -= d1[i + ci] - 1
    d1[i + ci] -= 1
    delta -= d2[i - ci + N - 1] - 1
    d2[i - ci + N - 1] -= 1
    # Remove (j, cj)
    delta -= d1[j + cj] - 1
    d1[j + cj] -= 1
    delta -= d2[j - cj + N - 1] - 1
    d2[j - cj + N - 1] -= 1
    # Add (i, cj)
    delta += d1[i + cj]
    d1[i + cj] += 1
    delta += d2[i - cj + N - 1]
    d2[i - cj + N - 1] += 1
    # Add (j, ci)
    delta += d1[j + ci]
    d1[j + ci] += 1
    delta += d2[j - ci + N - 1]
    d2[j - ci + N - 1] += 1
    board[i], board[j] = cj, ci
    return delta


def swap_delta(board, i, j, d1, d2, N):
    """Return the conflict-delta a swap of (i, j) WOULD cause, without
    leaving any side effect on board / d1 / d2. Uses apply_swap twice."""
    delta = apply_swap(board, i, j, d1, d2, N)
    apply_swap(board, i, j, d1, d2, N)
    return delta


def min_conflicts_repair(board, N, steps=20):
    """Local search on a permutation. At each step pick a row that lies on
    a conflicted diagonal and swap it with the row that most reduces total
    diagonal conflicts. Mutates board in place; returns final total.

    **Optimisation** — only OTHER conflicted rows are considered as swap
    partners (with a single non-conflicted fallback). This drops each step
    from O(N) to O(K), where K = number of currently conflicted rows. Once
    the chromosome is mostly clean, K is tiny so the repair cost goes to
    near-zero, which is what makes the memetic GA scale to N = 500.

    Used after crossover and mutation in the hybrid GA and as a warm start
    before SA at large N."""
    d1 = [0] * (2 * N)
    d2 = [0] * (2 * N)
    for r in range(N):
        c = board[r]
        d1[r + c] += 1
        d2[r - c + N - 1] += 1
    total = total_conflicts_perm(d1, d2)
    for _ in range(steps):
        if total == 0:
            break
        conflicted = [r for r in range(N)
                      if d1[r + board[r]] > 1 or d2[r - board[r] + N - 1] > 1]
        if not conflicted:
            break
        r1 = conflicted[0]
        # Fast path: prefer swapping with OTHER conflicted rows (O(K)).
        best_r2, best_delta = r1, 0
        for r2 in conflicted:
            if r2 == r1:
                continue
            delta = swap_delta(board, r1, r2, d1, d2, N)
            if delta < best_delta:
                best_delta = delta
                best_r2 = r2
        # Fallback: if no improvement among conflicted rows, scan ALL rows
        # once. This is the O(N) cost — but it only triggers near
        # convergence, when conflicts are sparse and an "outside" swap
        # partner is the only way to clear them.
        if best_r2 == r1:
            conflicted_set = set(conflicted)
            for r2 in range(N):
                if r2 == r1 or r2 in conflicted_set:
                    continue
                delta = swap_delta(board, r1, r2, d1, d2, N)
                if delta < best_delta:
                    best_delta = delta
                    best_r2 = r2
        if best_r2 == r1:
            break  # truly no improving swap available
        apply_swap(board, r1, best_r2, d1, d2, N)
        total += best_delta
    return total


## 1) Exhaustive DFS / Backtracking — with randomized restart

Plain depth-first backtracking with `cols`, `d1`, `d2` sets is exact but its worst-case
time grows factorially, and at `N=500` a single descent essentially never returns.

The fix is a **Las-Vegas wrapper around the same backtracker**: at each row the column
order is randomly shuffled, and a per-attempt backtrack budget is enforced. When the
budget is exhausted the search abandons the current tree and restarts with a fresh
random column order; the budget grows geometrically across restarts. The recursion
itself is identical to plain backtracking — sets `cols`, `(r+c) in d1`, `(r-c) in d2`
gate placements in O(1), and the first complete board (`r == N`) is returned.


In [ ]:
class _DFSTimeout(Exception):
    pass


class _DFSRestart(Exception):
    pass


def dfs_solve(N, timeout, seed=SEED):
    rng = random.Random(seed)
    cols = set()
    d1 = set()      # r + c
    d2 = set()      # r - c
    board = [-1] * N
    backtracks = [0]   # mutable counters for closure
    limit = [0]
    start = time.time()

    def backtrack(r):
        if time.time() - start > timeout:
            raise _DFSTimeout()
        if backtracks[0] >= limit[0]:
            raise _DFSRestart()
        if r == N:
            return True
        order = list(range(N))
        rng.shuffle(order)
        for c in order:
            if c in cols or (r + c) in d1 or (r - c) in d2:
                continue
            board[r] = c
            cols.add(c); d1.add(r + c); d2.add(r - c)
            if backtrack(r + 1):
                return True
            cols.discard(c); d1.discard(r + c); d2.discard(r - c)
            board[r] = -1
        backtracks[0] += 1
        return False

    old_limit = sys.getrecursionlimit()
    sys.setrecursionlimit(max(old_limit, N + 1000))
    base = max(N, 64)
    attempt = 0
    try:
        while True:
            if time.time() - start > timeout:
                return None, "timeout"
            attempt += 1
            # geometric growth: 1, 1, 1, 2, 2, 2, 4, 4, 4, 8, ...
            limit[0] = base * (1 << min((attempt - 1) // 3, 12))
            backtracks[0] = 0
            cols.clear(); d1.clear(); d2.clear()
            board[:] = [-1] * N
            try:
                if backtrack(0):
                    return board, 0
            except _DFSRestart:
                continue
    except _DFSTimeout:
        return None, "timeout"
    except RecursionError:
        return None, "recursion_error"
    finally:
        sys.setrecursionlimit(old_limit)


## 2) Hill Climbing with Min-Conflicts

For each step, pick a conflicted queen and move it to the column that minimises its conflicts.
Ties broken at random. Random restarts whenever the search plateaus or exhausts a step budget.
Conflict bookkeeping is incremental: O(N) per step for the column search, O(1) for accounting.


In [ ]:
def hill_climbing(N, timeout, seed=SEED):
    rng = random.Random(seed)
    start = time.time()
    best_board = None
    best_total = math.inf

    def queen_attacks(r, c, col, d1, d2):
        return (col[c] - 1) + (d1[r + c] - 1) + (d2[r - c + N - 1] - 1)

    plateau_limit = max(30, N // 2)
    max_steps = N * 20

    while time.time() - start < timeout:
        board = [rng.randrange(N) for _ in range(N)]
        col, d1, d2 = init_counts(board, N)
        total = total_conflicts(col, d1, d2)

        if total < best_total:
            best_total = total
            best_board = board[:]
        if total == 0:
            return board, 0

        plateau = 0
        for _ in range(max_steps):
            if time.time() - start > timeout:
                return best_board, best_total

            conflicted = [r for r in range(N)
                          if queen_attacks(r, board[r], col, d1, d2) > 0]
            if not conflicted:
                break
            r = rng.choice(conflicted)
            old_c = board[r]

            # Remove queen, track delta
            delta = -(col[old_c] - 1) - (d1[r + old_c] - 1) - (d2[r - old_c + N - 1] - 1)
            col[old_c] -= 1
            d1[r + old_c] -= 1
            d2[r - old_c + N - 1] -= 1

            # Search best new column (O(N))
            best_cost = None
            best_cands = []
            for c in range(N):
                cost = col[c] + d1[r + c] + d2[r - c + N - 1]
                if best_cost is None or cost < best_cost:
                    best_cost = cost
                    best_cands = [c]
                elif cost == best_cost:
                    best_cands.append(c)
            new_c = rng.choice(best_cands)

            delta += best_cost
            col[new_c] += 1
            d1[r + new_c] += 1
            d2[r - new_c + N - 1] += 1
            board[r] = new_c
            new_total = total + delta

            if new_total >= total:
                plateau += 1
            else:
                plateau = 0
            total = new_total

            if total < best_total:
                best_total = total
                best_board = board[:]
            if total == 0:
                return board, 0
            if plateau > plateau_limit:
                break  # random restart

    return best_board, best_total


## 3) Simulated Annealing

Permutation representation (board[r] is unique per r, so no column conflicts to track).
Neighbour move = swap of two rows. `apply_swap` mutates the diagonal counts in place and
returns the conflict delta in O(1), so each iteration is O(1). If the proposal is rejected,
applying the swap again exactly undoes it and returns `-delta`. Temperature cools
geometrically and reheats when it drops below `T_min`.

**Paper-inspired warm start.** For `N >= WARM_START_N` the random permutation is first
improved by `min_conflicts_repair`. This brings the initial conflict count from the
typical random-permutation level (≈ a few × N) down close to zero, so the Metropolis
loop spends its iterations on the hard tail of the search rather than the easy bulk.


In [ ]:
def simulated_annealing(N, timeout, seed=SEED, warm_start_n=WARM_START_N):
    rng = random.Random(seed)
    start = time.time()

    board = list(range(N))
    rng.shuffle(board)

    # Optional min-conflicts warm start for large N.
    if N >= warm_start_n:
        warm_steps = min(N, 500)
        min_conflicts_repair(board, N, steps=warm_steps)

    d1 = [0] * (2 * N)
    d2 = [0] * (2 * N)
    for r in range(N):
        c = board[r]
        d1[r + c] += 1
        d2[r - c + N - 1] += 1
    total = total_conflicts_perm(d1, d2)

    best_board = board[:]
    best_total = total
    if total == 0:
        return board, 0

    T = max(1.0, N / 2.0)
    T_min = 1e-3
    alpha = 0.9995
    reheat_at = T_min
    reheat_T = max(1.0, N / 4.0)

    # Conflict-biased neighbour selection: paper-inspired tweak that makes
    # SA tractable at large N once the warm start has pruned the bulk of the
    # conflicts. We recompute the conflicted-row cache every "refresh"
    # iterations because it goes stale as the board changes.
    conflicted_cache = []
    refresh = max(50, N // 4)
    bias_prob = 0.85

    max_iters = N * 5000
    for it in range(max_iters):
        if time.time() - start > timeout:
            break
        if total == 0:
            break

        if it % refresh == 0:
            conflicted_cache = [
                r for r in range(N)
                if d1[r + board[r]] > 1 or d2[r - board[r] + N - 1] > 1
            ]

        if conflicted_cache and rng.random() < bias_prob:
            i = rng.choice(conflicted_cache)
        else:
            i = rng.randrange(N)
        j = rng.randrange(N)
        if i == j:
            continue

        delta = apply_swap(board, i, j, d1, d2, N)
        if delta <= 0 or rng.random() < math.exp(-delta / T):
            total += delta
            if total < best_total:
                best_total = total
                best_board = board[:]
        else:
            # Undo: same swap reverses it
            apply_swap(board, i, j, d1, d2, N)

        T *= alpha
        if T < reheat_at:
            T = reheat_T

    return best_board, best_total


## 4) Genetic Algorithm (memetic / hybrid)

Permutation chromosomes, tournament selection, ordered crossover (OX), elitism. Lower
fitness (diagonal-conflict count) is better.

**Paper-inspired hybridisation.** The classical GA on N-Queens stalls a few queens
short of the optimum because random swap mutation is blind to the structure of the
conflicts. This implementation replaces plain mutation with a **conflict-aware
mutation** — a row that lies on a conflicted diagonal is selected, and it is swapped
with the partner that yields the largest local conflict reduction (random fallback to
keep diversity). After both crossover and mutation, the child is passed through
`min_conflicts_repair(child, steps=20)`, turning the algorithm into a memetic GA while
preserving GA scaffolding (tournament selection, OX, elitism).


In [ ]:
def genetic_algorithm(N, timeout, seed=SEED):
    rng = random.Random(seed)
    start = time.time()

    POP = 120
    ELITE = 10
    TOURN = 5
    MUT_RATE = 0.4
    REPAIR_STEPS = 20

    def fitness(board):
        d1 = [0] * (2 * N)
        d2 = [0] * (2 * N)
        for r in range(N):
            c = board[r]
            d1[r + c] += 1
            d2[r - c + N - 1] += 1
        s = 0
        for k in d1:
            if k > 1:
                s += k * (k - 1) // 2
        for k in d2:
            if k > 1:
                s += k * (k - 1) // 2
        return s

    def make_individual():
        b = list(range(N))
        rng.shuffle(b)
        return b

    def tournament(pop_fit):
        cands = rng.sample(pop_fit, TOURN)
        return min(cands, key=lambda x: x[1])[0]

    def ordered_crossover(p1, p2):
        a, b = sorted(rng.sample(range(N), 2))
        child = [-1] * N
        child[a:b + 1] = p1[a:b + 1]
        present = set(child[a:b + 1])
        idx = (b + 1) % N
        for gene in p2[b + 1:] + p2[:b + 1]:
            if gene not in present:
                child[idx] = gene
                idx = (idx + 1) % N
        return child

    def conflict_aware_mutate(board):
        """Pick a row on a conflicted diagonal; swap it with the partner that
        most reduces total diagonal conflicts (random fallback for diversity)."""
        if rng.random() >= MUT_RATE:
            return board
        d1 = [0] * (2 * N)
        d2 = [0] * (2 * N)
        for r in range(N):
            c = board[r]
            d1[r + c] += 1
            d2[r - c + N - 1] += 1
        conflicted = [r for r in range(N)
                      if d1[r + board[r]] > 1 or d2[r - board[r] + N - 1] > 1]
        if not conflicted:
            # No conflict to target — small random swap for diversity
            i, j = rng.sample(range(N), 2)
            board[i], board[j] = board[j], board[i]
            return board
        r1 = rng.choice(conflicted)
        best_r2, best_delta = -1, 0
        for r2 in range(N):
            if r2 == r1:
                continue
            delta = swap_delta(board, r1, r2, d1, d2, N)
            if delta < best_delta:
                best_delta = delta
                best_r2 = r2
        if best_r2 < 0:
            # No improving swap — random swap to escape local optimum
            candidates = [r for r in range(N) if r != r1]
            best_r2 = rng.choice(candidates)
        board[r1], board[best_r2] = board[best_r2], board[r1]
        return board

    population = [make_individual() for _ in range(POP)]
    pop_fit = [(ind, fitness(ind)) for ind in population]
    pop_fit.sort(key=lambda x: x[1])
    best_board = pop_fit[0][0][:]
    best_fit = pop_fit[0][1]
    if best_fit == 0:
        return best_board, 0

    while best_fit > 0:
        if time.time() - start > timeout:
            break
        new_pop = [ind[:] for ind, _ in pop_fit[:ELITE]]
        while len(new_pop) < POP:
            if time.time() - start > timeout:
                break
            p1 = tournament(pop_fit)
            p2 = tournament(pop_fit)
            child = ordered_crossover(p1, p2)
            min_conflicts_repair(child, N, steps=REPAIR_STEPS)
            child = conflict_aware_mutate(child)
            min_conflicts_repair(child, N, steps=REPAIR_STEPS)
            new_pop.append(child)
        pop_fit = [(ind, fitness(ind)) for ind in new_pop]
        pop_fit.sort(key=lambda x: x[1])
        if pop_fit[0][1] < best_fit:
            best_fit = pop_fit[0][1]
            best_board = pop_fit[0][0][:]
            if best_fit == 0:
                return best_board, 0

    return best_board, best_fit


## Benchmark runner

For each (algorithm, N) we measure wall time with `time.time()` and peak Python memory
with `tracemalloc`. The result categorisation:

- `Solved = "Yes"` when conflicts == 0
- `Solved = "Timeout"` when the algorithm exhausted its time budget
- `Solved = "No"` when DFS exhausted the search space without finding a solution

Results are written to `results.csv`.


In [ ]:
def benchmark(name, fn, N, timeout):
    gc.collect()
    tracemalloc.start()
    t0 = time.time()
    board, conflicts = fn(N, timeout, SEED)
    elapsed = time.time() - t0
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    if conflicts == 0:
        solved = "Yes"
        final = 0
    elif conflicts == "timeout":
        solved = "Timeout"
        final = "Timeout"
    elif conflicts == "no_solution":
        solved = "No"
        final = "No solution"
    elif elapsed >= timeout * 0.95:
        solved = "Timeout"
        final = conflicts
    else:
        solved = "No"
        final = conflicts

    return {
        "Algorithm": name,
        "N": N,
        "Solved": solved,
        "FinalConflicts": final,
        "TimeSeconds": round(elapsed, 4),
        "PeakMemoryMB": round(peak / (1024 * 1024), 4),
    }


ALGORITHMS = [
    ("DFS Backtracking", dfs_solve),
    ("Hill Climbing", hill_climbing),
    ("Simulated Annealing", simulated_annealing),
    ("Genetic Algorithm", genetic_algorithm),
]

results = []
for N in N_VALUES:
    print(f"=== N={N} ===")
    for name, fn in ALGORITHMS:
        print(f"  {name}...", end=" ", flush=True)
        row = benchmark(name, fn, N, TIMEOUT)
        results.append(row)
        print(f"solved={row['Solved']:>7} conflicts={row['FinalConflicts']!s:>9} "
              f"time={row['TimeSeconds']:>8.4f}s mem={row['PeakMemoryMB']:.3f}MB")

df = pd.DataFrame(results, columns=["Algorithm", "N", "Solved", "FinalConflicts",
                                    "TimeSeconds", "PeakMemoryMB"])
df.to_csv("results.csv", index=False)
df


## Plots

Three comparison plots are written: `time_comparison.png`, `memory_comparison.png`, and
`conflicts_comparison.png`. Non-numeric conflict values (`"Timeout"`, `"No solution"`) are
omitted from the conflicts plot.


In [ ]:
def plot_metric(df, metric, ylabel, title, fname, log=False):
    plt.figure(figsize=(10, 6))
    for algo in df["Algorithm"].unique():
        sub = df[df["Algorithm"] == algo].copy()
        sub["_y"] = pd.to_numeric(sub[metric], errors="coerce")
        sub = sub.dropna(subset=["_y"])
        if sub.empty:
            continue
        plt.plot(sub["N"], sub["_y"], marker="o", label=algo)
    plt.xlabel("N")
    plt.ylabel(ylabel)
    plt.title(title)
    if log:
        plt.yscale("log")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(fname, dpi=120)
    plt.show()


plot_metric(df, "TimeSeconds", "Time (seconds, log scale)",
            "Time vs N", "time_comparison.png", log=True)
plot_metric(df, "PeakMemoryMB", "Peak Memory (MB)",
            "Peak Memory vs N", "memory_comparison.png")
plot_metric(df, "FinalConflicts", "Final Conflicts",
            "Final Conflicts vs N", "conflicts_comparison.png")

print("Wrote results.csv, time_comparison.png, memory_comparison.png, conflicts_comparison.png")
